In [2]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Importação dos Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

df = pd.read_csv('Dataset/earthquakes_cleaned.csv')

In [3]:
df_model = df.copy()
df_model['alert'] = df_model['alert'].replace('Unknown', 'No_Alert')

# 2. Separar Features (X) e Target (y)
colunas_para_remover = ['alert', 'date', 'ano_mes']
X = df_model.drop(columns=colunas_para_remover)
y = df_model['alert']

# 3. Codificar a variável alvo (Target) para números
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Visualizar o mapeamento das classes
classes_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(f"Mapeamento das classes alvo: {classes_mapping}")

# 4. Separar colunas numéricas e categóricas
num_features = X.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 5. Criar o pré-processador para os modelos clássicos
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ])

# 6. Divisão em Treino e Teste (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Amostras de Treino: {X_train.shape[0]} | Amostras de Teste: {X_test.shape[0]}")

Mapeamento das classes alvo: {'No_Alert': np.int64(0), 'green': np.int64(1), 'orange': np.int64(2), 'red': np.int64(3), 'yellow': np.int64(4)}
Amostras de Treino: 481 | Amostras de Teste: 121


C:\Users\julio\AppData\Local\Temp\ipykernel_39776\2730606236.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()


In [4]:
# Configurar os modelos que precisam de pré-processamento
# Usamos class_weight='balanced' onde possível para compensar o domínio da classe 'No_Alert'
# Modelos do Scikit-Learn continuam na CPU.
# XGBoost e LightGBM configurados para buscar a GPU.
pipeline_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    # Ativando GPU no XGBoost
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42, device='cuda'),
    # Ativando GPU no LightGBM
    "LightGBM": LGBMClassifier(random_state=42, class_weight='balanced', verbose=-1, device='gpu'),
    "SVC (Support Vector)": SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = {}
trained_models = {}

print("Iniciando treinamento dos modelos em Pipeline...\n")

for name, model in pipeline_models.items():
    print(f"--- Treinando {name} ---")

    # Criar e treinar o pipeline
    clf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    clf.fit(X_train, y_train)

    # Prever e avaliar
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Salvar resultados e pipeline
    results[name] = {'Accuracy': acc, 'F1-Score (Weighted)': f1}
    trained_models[name] = clf
    print(f"F1-Score: {f1:.4f}\n")

Iniciando treinamento dos modelos em Pipeline...

--- Treinando Logistic Regression ---
F1-Score: 0.9615

--- Treinando Random Forest ---
F1-Score: 0.9605

--- Treinando XGBoost ---


C:\Users\julio\PycharmProjects\Earthquake_Model\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:22:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\julio\PycharmProjects\Earthquake_Model\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [23:22:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


F1-Score: 0.9707

--- Treinando LightGBM ---
F1-Score: 0.9707

--- Treinando SVC (Support Vector) ---
F1-Score: 0.9681

--- Treinando KNN ---
F1-Score: 0.9308



C:\Users\julio\PycharmProjects\Earthquake_Model\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
# O CatBoost usa os dados originais (sem OneHotEncoding) e trata as categorias internamente
print("--- Treinando CatBoost na GPU ---")

cb_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    cat_features=cat_features,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=0,
    task_type='GPU' # <-- Parâmetro mágico para ativar a GPU no CatBoost
)

# Treinamento com validação cruzada simples (early stopping)
cb_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

# Prever e avaliar
y_pred_cb = cb_model.predict(X_test)
acc_cb = accuracy_score(y_test, y_pred_cb)
f1_cb = f1_score(y_test, y_pred_cb, average='weighted')

# Salvar resultados
results["CatBoost"] = {'Accuracy': acc_cb, 'F1-Score (Weighted)': f1_cb}
trained_models["CatBoost"] = cb_model
print(f"F1-Score CatBoost: {f1_cb:.4f}\n")

--- Treinando CatBoost na GPU ---
F1-Score CatBoost: 0.9829



In [6]:
# 1. Tabela de Comparação
print("=== Comparação Final de Todos os Modelos ===")
results_df = pd.DataFrame(results).T
display(results_df.sort_values(by='F1-Score (Weighted)', ascending=False))

# 2. Criar diretório para salvar
output_dir = 'Trained_Models'
os.makedirs(output_dir, exist_ok=True)

# 3. Salvar todos os modelos
for name, model in trained_models.items():
    if name == "CatBoost":
        # CatBoost tem um formato próprio otimizado
        filepath = os.path.join(output_dir, 'CatBoost_model.cbm')
        model.save_model(filepath)
    else:
        # Outros modelos são salvos via joblib
        filename = f"{name.replace(' ', '_')}_pipeline.pkl"
        filepath = os.path.join(output_dir, filename)
        joblib.dump(model, filepath)

# 4. Salvar o LabelEncoder (Obrigatório para reverter previsões futuras para texto)
joblib.dump(label_encoder, os.path.join(output_dir, 'target_label_encoder.pkl'))

print("\nTodos os modelos e o codificador de alertas foram salvos com sucesso no diretório 'Trained_Models'!")

=== Comparação Final de Todos os Modelos ===


,Accuracy,F1-Score (Weighted)
CatBoost,0.983471,0.982949
LightGBM,0.975207,0.970694
XGBoost,0.975207,0.970694
SVC (Support Vector),0.966942,0.968129
Logistic Regression,0.958678,0.961473
Random Forest,0.966942,0.960462
KNN,0.933884,0.930815



Todos os modelos e o codificador de alertas foram salvos com sucesso no diretório 'Trained_Models'!
